In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-02-28 00:15:03 | people | execute | Started
2025-02-28 00:15:03 | people | load | Started
2025-02-28 00:15:07 | people | load | Completed in 0.07 min
2025-02-28 00:15:07 | people | transform | Started
2025-02-28 00:15:07 | people | transform | Completed in 0.0 min
2025-02-28 00:15:07 | people | write | Started
2025-02-28 00:15:48 | people | write | Completed in 0.67 min
2025-02-28 00:15:48 | people | execute | Completed in 0.75 min
2025-02-28 00:15:48 | planets | execute | Started
2025-02-28 00:15:48 | planets | load | Started
2025-02-28 00:15:52 | planets | load | Completed in 0.05 min
2025-02-28 00:15:52 | planets | transform | Started
2025-02-28 00:15:52 | planets | transform | Completed in 0.0 min
2025-02-28 00:15:52 | planets | write | Started
2025-02-28 00:16:34 | planets | write | Completed in 0.68 min
2025-02-28 00:16:34 | planets | execute | Completed in 0.75 min


In [10]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-02-28 00:15:...|      Luke Skywalker|  1|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:15:...|               C-3PO|  2|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:15:...|               R2-D2|  3|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:15:...|         Darth Vader|  4|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:15:...|         Leia Organa|  5|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:15:...|           Owen Lars|  6|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:15:...|  Beru Whitesun lars|  7|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:15:...|               R5-D4|  8|https://www.swapi...|{"created": "2025..

In [11]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(truncate=False)

No. Rows: 60
+--------------------------+--------------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+--------------+---+--

In [12]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [13]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [14]:
config = {
    "load": {
        "mode": "default",
        "filter": "all",
        "date_col": "LH_BronzeTS",
    },
    "transform": {
        "ignore_defaults": False,
        "transformation_order": [
            "add_dummy_col",
            "rename_columns",
            "tbl_transformations",
            "select_columns",
            "cast_column_types",
        ],
        # "tbl_transformations": {"tbl": "custom_transform1"},
        "rename_columns": {
            "planets": {"dummy_col": "dummy"},
            "people": {"dummy_col": "dummy"},
        },
        "select_columns": {
            "planets": ["LH_BronzeTS", "name", "uid", "url", "dummy"],
            "people": ["LH_BronzeTS", "name", "uid", "url", "dummy"],
        },
        "cast_column_types": {
            "planets": {"dummy": "string", "id": "int"},
            "people": {"dummy": "string", "id": "int"},
        },
    },
    "write": {
        "mode": "overwrite",
        "merge_schema": True,
        "external": False,
    },
    "optimize": {
        "optimize": True,
        "optimize_full": False,
        "vacuum": True,
        "vacuum_lite": True,
        "analyze": False,
        "retention": 168,
        # "excl_cols": ["a", "b"],
    },
    "tblproperties": {
        # "clusterby": ["a", "b"],
        "deletion_vectors": True,
        "auto_compact": True,
        "optimize_write": True,
        "change_data_feed": True,
        "row_tracking": True,
        "type_widening": False,
        "tblproperties": {"enableChangeDataFeed": "true"},
    },
}

In [15]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df

    def add_dummy_col(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("dummy_col", F.lit("dummy"))


silver_instance = StarWarsSilver(
    spark,
    catalog=CATALOG,
    source_schema="bronze",
    target_schema="silver",
    config=config,
)

In [16]:
silver_instance.execute("people", "planets")

2025-02-28 00:16:38 | people | execute | Started
2025-02-28 00:16:38 | people | load | Started
2025-02-28 00:16:38 | people | load | Completed in 0.0 min
2025-02-28 00:16:38 | people | transform | Started
2025-02-28 00:16:38 | people | transform | Completed in 0.0 min
2025-02-28 00:16:38 | people | write | Started
2025-02-28 00:16:40 | people | write | Completed in 0.03 min
2025-02-28 00:16:40 | people | tblproperties | Started
2025-02-28 00:16:55 | people | tblproperties | Completed in 0.23 min
2025-02-28 00:16:55 | people | optimize | Started
2025-02-28 00:17:08 | people | optimize | Completed in 0.2 min
2025-02-28 00:17:08 | people | execute | Completed in 0.48 min
2025-02-28 00:17:08 | planets | execute | Started
2025-02-28 00:17:08 | planets | load | Started
2025-02-28 00:17:08 | planets | load | Completed in 0.0 min
2025-02-28 00:17:08 | planets | transform | Started
2025-02-28 00:17:08 | planets | transform | Completed in 0.0 min
2025-02-28 00:17:08 | planets | write | Started
2

In [17]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 82
+-------------------------+--------------------------+---------------------+---+------------------------------------+-----+
|LH_SilverTS              |LH_BronzeTS               |name                 |uid|url                                 |dummy|
+-------------------------+--------------------------+---------------------+---+------------------------------------+-----+
|2025-02-28 00:16:38.61919|2025-02-28 00:15:08.662301|Cliegg Lars          |62 |https://www.swapi.tech/api/people/62|dummy|
|2025-02-28 00:16:38.61919|2025-02-28 00:15:08.662301|Poggle the Lesser    |63 |https://www.swapi.tech/api/people/63|dummy|
|2025-02-28 00:16:38.61919|2025-02-28 00:15:08.662301|Luminara Unduli      |64 |https://www.swapi.tech/api/people/64|dummy|
|2025-02-28 00:16:38.61919|2025-02-28 00:15:08.662301|Barriss Offee        |65 |https://www.swapi.tech/api/people/65|dummy|
|2025-02-28 00:16:38.61919|2025-02-28 00:15:08.662301|Dormé                |66 |https://www.swapi.tech/api/people/66|du

In [18]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 60
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----+
|LH_SilverTS               |LH_BronzeTS               |name          |uid|url                                  |dummy|
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----+
|2025-02-28 00:17:08.089801|2025-02-28 00:15:52.166954|Mygeeto       |16 |https://www.swapi.tech/api/planets/16|dummy|
|2025-02-28 00:17:08.089801|2025-02-28 00:15:52.166954|Felucia       |17 |https://www.swapi.tech/api/planets/17|dummy|
|2025-02-28 00:17:08.089801|2025-02-28 00:15:52.166954|Cato Neimoidia|18 |https://www.swapi.tech/api/planets/18|dummy|
|2025-02-28 00:17:08.089801|2025-02-28 00:15:52.166954|Saleucami     |19 |https://www.swapi.tech/api/planets/19|dummy|
|2025-02-28 00:17:08.089801|2025-02-28 00:15:52.166954|Stewjon       |20 |https://www.swapi.tech/api/planets/20|dummy|
|2025-02-28 00:17:08.089801|2025-02

In [19]:
df = spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.planets")
df.show(100, truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                                                                                                                                             |job |notebook|clusterId|readVersion|isolationLevel   |isBlindAppend|op

# 6 Clean Up

In [20]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]